# 1) Imports básicos


In [1]:
import io
import numpy as np
import pandas as pd
from google.colab import files
from sklearn.linear_model import LinearRegression

# 2) Upload do CSV de sales_items


In [2]:
sales_items = pd.read_csv('dataset_fashion_store_salesitems.csv')
sales_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2253 entries, 0 to 2252
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   item_id            2253 non-null   int64  
 1   sale_id            2253 non-null   int64  
 2   product_id         2253 non-null   int64  
 3   quantity           2253 non-null   int64  
 4   original_price     2253 non-null   float64
 5   unit_price         2253 non-null   float64
 6   discount_applied   2253 non-null   float64
 7   discount_percent   2253 non-null   object 
 8   discounted         2253 non-null   int64  
 9   item_total         2253 non-null   float64
 10  sale_date          2253 non-null   object 
 11  channel            2253 non-null   object 
 12  channel_campaigns  2253 non-null   object 
dtypes: float64(4), int64(5), object(4)
memory usage: 228.9+ KB


# 3) Garantir que a coluna de data está como datetime


In [3]:
sales_items['sale_date'] = pd.to_datetime(sales_items['sale_date'])

# 4) Definir a coluna de valor da linha


In [4]:
sales_items['line_value'] = sales_items['item_total']

sales_items[['sale_date', 'line_value']].head()

,sale_date,line_value
0,2025-06-16,81.80
1,2025-06-17,81.79
2,2025-04-16,80.76
3,2025-05-06,78.52
4,2025-06-15,78.52


# 5) Agregar faturamento diário


In [5]:
daily_raw = (
    sales_items
    .groupby(["sale_date", "channel"], as_index=False)["line_value"]
    .sum()
)

daily_raw.columns = ["sale_date", "channel", "total_sales"]

daily_raw = daily_raw.sort_values(["channel", "sale_date"])
daily_raw.head()

,sale_date,channel,total_sales
0,2025-04-04,App Mobile,2195.06
2,2025-04-05,App Mobile,2589.47
4,2025-04-06,App Mobile,2182.46
6,2025-04-13,App Mobile,2559.39
8,2025-04-14,App Mobile,1578.26


## 6) Criar um calendário contínuo de datas


In [6]:
channels = sorted(daily_raw["channel"].dropna().unique())

daily_parts = []

for ch in channels:
    daily_ch_raw = daily_raw[daily_raw["channel"] == ch].copy()

    full_dates = pd.DataFrame({
        "sale_date": pd.date_range(
            start=daily_ch_raw["sale_date"].min(),
            end=daily_ch_raw["sale_date"].max(),
            freq="D"
        )
    })

    daily_ch = full_dates.merge(daily_ch_raw, on="sale_date", how="left")
    daily_ch["channel"] = daily_ch["channel"].fillna(ch)
    daily_ch["total_sales"] = daily_ch["total_sales"].fillna(0)

    daily_parts.append(daily_ch)

daily = pd.concat(daily_parts, ignore_index=True)
daily = daily.sort_values(["channel", "sale_date"]).reset_index(drop=True)
daily.head()


,sale_date,channel,total_sales
0,2025-04-04,App Mobile,2195.06
1,2025-04-05,App Mobile,2589.47
2,2025-04-06,App Mobile,2182.46
3,2025-04-07,App Mobile,0.00
4,2025-04-08,App Mobile,0.00


# 7) Criar índice temporal e treinar regressão linear simples


In [7]:
from sklearn.linear_model import LinearRegression
import numpy as np

models = {}

for ch in channels:
    df_ch = daily[daily['channel'] == ch].copy()
    df_ch['t'] = np.arange(1, len(df_ch) + 1)
    daily.loc[daily['channel'] == ch, 't'] = df_ch['t'].values

    X = df_ch[['t']].values
    y = df_ch['total_sales'].values

    m = LinearRegression()
    m.fit(X, y)
    models[ch] = (m, len(df_ch))

# 8) Gerar previsão para os próximos N dias


In [8]:
n_forecast_days = 7

future_parts = []

for ch in channels:
    m, n_hist = models[ch]

    last_date_ch = daily[daily['channel'] == ch]['sale_date'].max()

    future_t = np.arange(n_hist + 1, n_hist + 1 + n_forecast_days)
    future_dates_ch = pd.date_range(
        start=last_date_ch + pd.Timedelta(days=1),
        periods=n_forecast_days,
        freq='D'
    )
    future_sales_ch = m.predict(future_t.reshape(-1, 1))

    future_parts.append(pd.DataFrame({
        'sale_date': future_dates_ch,
        'channel': ch,
        'total_sales': np.nan,
        'forecast_sales': future_sales_ch
    }))

future = pd.concat(future_parts, ignore_index=True)

# 9) Montar tabela final histórico + previsão


In [9]:
hist = daily[['sale_date', 'channel', 'total_sales']].copy()
hist['forecast_sales'] = np.nan

forecast_df = pd.concat([hist, future], ignore_index=True)
forecast_df = forecast_df.sort_values(['channel', 'sale_date']).reset_index(drop=True)
forecast_df.tail(10)

,sale_date,channel,total_sales,forecast_sales
154,2025-06-15,E-commerce,3279.05,NaN
155,2025-06-16,E-commerce,3597.82,NaN
156,2025-06-17,E-commerce,3034.13,NaN
157,2025-06-18,E-commerce,NaN,2268.371005
158,2025-06-19,E-commerce,NaN,2267.827884
159,2025-06-20,E-commerce,NaN,2267.284764
160,2025-06-21,E-commerce,NaN,2266.741643
161,2025-06-22,E-commerce,NaN,2266.198522
162,2025-06-23,E-commerce,NaN,2265.655401
163,2025-06-24,E-commerce,NaN,2265.112280


# 10) Exportar CSV para usar no Power BI


In [10]:
forecast_df.to_csv('forecast_sales.csv', index=False)

files.download('forecast_sales.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>